In [4]:
import pandas as pd

df = pd.read_parquet("data/raw/yellow_tripdata_2025-01.parquet")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print()
print(df.dtypes)

Rows: 3,475,226
Columns: 20

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object


In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [6]:
n = len(df)

def pct(count):
    return f"{count:>10,}  ({count/n*100:6.3f}%)"

print(f"Total rows: {n:,}\n")

print("Dropoff before pickup: ", pct((df.tpep_dropoff_datetime < df.tpep_pickup_datetime).sum()))
print("Zero-length trips:     ", pct((df.tpep_dropoff_datetime == df.tpep_pickup_datetime).sum()))
print("Outside January 2025:  ", pct((~df.tpep_pickup_datetime.between('2025-01-01', '2025-02-01')).sum()))
print("Negative fare:         ", pct((df.fare_amount < 0).sum()))
print("Zero fare:             ", pct((df.fare_amount == 0).sum()))
print("Negative total:        ", pct((df.total_amount < 0).sum()))
print("Zero distance:         ", pct((df.trip_distance == 0).sum()))
print("Distance > 100 miles:  ", pct((df.trip_distance > 100).sum()))
print("Passenger count null:  ", pct(df.passenger_count.isna().sum()))
print("Passenger count = 0:   ", pct((df.passenger_count == 0).sum()))
print("Bad pickup zone:       ", pct((~df.PULocationID.between(1, 265)).sum()))
print("Bad dropoff zone:      ", pct((~df.DOLocationID.between(1, 265)).sum()))

Total rows: 3,475,226

Dropoff before pickup:         124  ( 0.004%)
Zero-length trips:           1,927  ( 0.055%)
Outside January 2025:           22  ( 0.001%)
Negative fare:             144,118  ( 4.147%)
Zero fare:                   1,398  ( 0.040%)
Negative total:             63,037  ( 1.814%)
Zero distance:              90,893  ( 2.615%)
Distance > 100 miles:          162  ( 0.005%)
Passenger count null:      540,149  (15.543%)
Passenger count = 0:        24,656  ( 0.709%)
Bad pickup zone:                 0  ( 0.000%)
Bad dropoff zone:                0  ( 0.000%)


In [7]:
neg = df[df.fare_amount < 0]

print("=== NEGATIVE FARES ===")
print(f"Count: {len(neg):,}\n")
print("By payment_type:")
print(neg.payment_type.value_counts())
print("\nBy VendorID:")
print(neg.VendorID.value_counts())
print("\nFare summary:")
print(neg.fare_amount.describe())
print("\nSample rows:")
print(neg[['VendorID','payment_type','trip_distance','fare_amount','total_amount']].head(10))

=== NEGATIVE FARES ===
Count: 144,118

By payment_type:
payment_type
0    84822
4    37405
2    13802
3     8075
1       14
Name: count, dtype: int64

By VendorID:
VendorID
2    144118
Name: count, dtype: int64

Fare summary:
count    144118.000000
mean        -11.630930
std          19.384895
min        -900.000000
25%         -10.700000
50%          -4.750000
75%          -4.750000
max          -0.010000
Name: fare_amount, dtype: float64

Sample rows:
     VendorID  payment_type  trip_distance  fare_amount  total_amount
17          2             2           0.71         -7.2         -8.54
22          2             4           0.69         -6.5        -11.50
104         2             4           0.97        -16.3        -21.30
149         2             2           1.42        -12.1        -17.10
202         2             4           0.60         -7.2        -12.20
212         2             4           1.88        -14.2        -19.20
364         2             4           0.01         -

In [8]:
weird = df[(df.fare_amount < 0) & (df.total_amount >= 0)]

print(f"Negative fare, non-negative total: {len(weird):,}\n")
print("By payment_type:")
print(weird.payment_type.value_counts())
print("\nTotal amount summary:")
print(weird.total_amount.describe())
print("\nSample:")
print(weird[['payment_type','trip_distance','fare_amount','extra','tolls_amount','total_amount']].head(10))

Negative fare, non-negative total: 81,402

By payment_type:
payment_type
0    81401
4        1
Name: count, dtype: int64

Total amount summary:
count    81402.000000
mean         4.007227
std          2.402513
min          0.010000
25%          2.690000
50%          3.480000
75%          5.000000
max         60.000000
Name: total_amount, dtype: float64

Sample:
         payment_type  trip_distance  fare_amount  extra  tolls_amount  total_amount
297339              4           0.78        -9.30    0.0          0.00          1.70
2935080             0           3.55        -2.35    0.0          0.00          5.80
2935098             0           7.30        -0.47    0.0          0.00          3.53
2935115             0           0.76        -2.27    0.0          0.00          1.73
2935124             0           4.99        -0.19    0.0          0.00          3.81
2935133             0          13.12        -1.29    0.0          6.94          7.15
2935151             0           1.95     

In [9]:
p0 = df[df.payment_type == 0]

print(f"payment_type = 0 rows: {len(p0):,}\n")
print("Date range:")
print(p0.tpep_pickup_datetime.min(), "to", p0.tpep_pickup_datetime.max())
print("\nBy VendorID:")
print(p0.VendorID.value_counts())
print("\nRatecodeID:")
print(p0.RatecodeID.value_counts(dropna=False))
print("\nPassenger count nulls in this group:", p0.passenger_count.isna().sum())
print("Total payment_type=0 rows:", len(p0))

payment_type = 0 rows: 540,149

Date range:
2025-01-01 00:00:00 to 2025-01-31 23:59:59

By VendorID:
VendorID
2    451456
1     88204
6       489
Name: count, dtype: int64

RatecodeID:
RatecodeID
NaN    540149
Name: count, dtype: int64

Passenger count nulls in this group: 540149
Total payment_type=0 rows: 540149


In [10]:
p0 = df[df.payment_type == 0]
rest = df[df.payment_type != 0]

print("Nulls in payment_type=0 group:")
print(p0.isna().sum()[p0.isna().sum() > 0])

print("\nNulls in everything else:")
print(rest.isna().sum()[rest.isna().sum() > 0])

print("\nStore-and-forward flag:")
print("  p0:  ", p0.store_and_fwd_flag.value_counts(dropna=False).to_dict())
print("  rest:", rest.store_and_fwd_flag.value_counts(dropna=False).to_dict())

print("\nTrip distance — p0 vs rest:")
print(p0.trip_distance.describe())
print(rest.trip_distance.describe())

Nulls in payment_type=0 group:
passenger_count         540149
RatecodeID              540149
store_and_fwd_flag      540149
congestion_surcharge    540149
Airport_fee             540149
dtype: int64

Nulls in everything else:
Series([], dtype: int64)

Store-and-forward flag:
  p0:   {None: 540149}
  rest: {'N': 2927431, 'Y': 7646}

Trip distance — p0 vs rest:
count    540149.000000
mean         20.314573
std        1428.575951
min           0.000000
25%           1.130000
50%           2.040000
75%           3.610000
max      276423.570000
Name: trip_distance, dtype: float64
count    2.935077e+06
mean     3.194121e+00
std      4.261888e+01
min      0.000000e+00
25%      9.600000e-01
50%      1.600000e+00
75%      2.990000e+00
max      4.473030e+04
Name: trip_distance, dtype: float64
